# **Grocery Items Object Detection with Automatic Labelling**


Pangilinan, Reignel Bernice A.

2018-01460

## **Load and process the dataset from the class GDrive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
import os

# Define paths
dataset_root = "/content/drive/MyDrive/AI 231 ME4 Dataset/"
yolo_dataset_root = "/content/yolo_dataset"

# Create YOLO folders
# Train dataset
os.makedirs(f"{yolo_dataset_root}/images/train", exist_ok=True)
os.makedirs(f"{yolo_dataset_root}/labels/train", exist_ok=True)

# Test dataset
os.makedirs(f"{yolo_dataset_root}/images/test", exist_ok=True)
os.makedirs(f"{yolo_dataset_root}/labels/test", exist_ok=True)

In [ ]:
# Collect and merge train/test data into "train" and "test" folders
import shutil
from glob import glob

def collect_split_data(class_root, split):
    img_dir = os.path.join(class_root, split, "images")
    lbl_dir = os.path.join(class_root, split, "labels")

    if not os.path.exists(img_dir):
        return

    for img_path in glob(os.path.join(img_dir, "*")):
        if img_path.lower().endswith((".jpg", ".jpeg", ".png")):
            file_name = os.path.basename(img_path)
            lbl_path = os.path.join(lbl_dir, os.path.splitext(file_name)[0] + ".txt")

            dest_split = "train" if split == "train" else "test"
            shutil.copy(img_path, os.path.join(yolo_dataset_root, f"images/{dest_split}", file_name))

            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, os.path.join(yolo_dataset_root, f"labels/{dest_split}", os.path.basename(lbl_path)))

# Loop through all product folders
for class_name in os.listdir(dataset_root):
    class_path = os.path.join(dataset_root, class_name)
    if not os.path.isdir(class_path):
        continue
    collect_split_data(class_path, "train")
    collect_split_data(class_path, "test")

print("✅ Combined train/test data into YOLO dataset folders.")

KeyboardInterrupt: 

In [ ]:
# Create data.yaml

import yaml

classes = sorted(os.listdir(dataset_root))
dataset_yaml = {
    'path': yolo_dataset_root,
    'train': 'images/train',
    'test': 'images/test',
    'names': {i: cls for i, cls in enumerate(classes)}
}

with open(f'{yolo_dataset_root}/data.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)

print("✅ Created data.yaml file for YOLO")

## **Fine-tune YOLO model**

In [ ]:
!pip install ultralytics gradio --quiet

import ultralytics
ultralytics.checks()

Ultralytics 8.3.205 🚀 Python-3.12.11 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.6/107.7 GB disk)


In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO model
model = YOLO('yolov8n.pt')  # or yolov11n.pt if you prefer the latest

# Fine-tune with your dataset
model.train(
    data=f'{yolo_dataset_root}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='product_yolo_finetuned'
)

# fine-tuned model will be saved to 'runs/detect/product_yolo_finetuned/weights/best.pt'

## **Gradio interface for inference**

In [ ]:
import gradio as gr
from ultralytics import YOLO
import cv2
import numpy as np

# Load your fine-tuned YOLO model
model = YOLO("runs/detect/product_yolo_finetuned/weights/best.pt")

# Define YOLO prediction function for each video frame
def predict_frame(frame):
    # Run YOLO inference on the frame
    results = model.predict(source=frame, conf=0.4, verbose=False)

    # Get annotated frame with bounding boxes
    annotated_frame = results[0].plot()

    # Convert from BGR to RGB for Gradio display
    annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
    return annotated_frame


# Create Gradio interface using webcam (live video)
demo = gr.Interface(
    fn=predict_frame,
    inputs=gr.Image(
        source="webcam",
        streaming=True,
        label="Webcam Feed"
    ),
    outputs=gr.Image(label="Detection Output"),
    title="YOLO Product Detection (Webcam)",
    description="Live object detection using a fine-tuned YOLO model. Shows bounding boxes in real time.",
    live=True,
)

# Launch app (share=True if you want a public link)
demo.launch(share=True)